# Colab setup (run top to bottom)
1. Runtime > Change runtime type > T4 GPU.
2. Change YOUR_NAME below, then run each cell in order.
3. You will be asked once to paste your Kaggle API token (never saved in this notebook).
4. Data is downloaded from Kaggle into the Colab VM (not git). It is wiped when the session ends; just re-run.

In [1]:
import os
from getpass import getpass
os.environ['RUN_USER'] = 'YOUR_NAME'   # <- change this (labels your experiment runs)

# 1) code: public GitHub repo, no token needed
!rm -rf /content/repo && git clone -q https://github.com/JerNB/daml-indoor67.git /content/repo
%cd /content/repo
!pip -q install timm kaggle
import torch; print('GPU available:', torch.cuda.is_available())

/content/repo
GPU available: True


In [ ]:
# 2) data: download from Kaggle (your account must have joined the competition and accepted the rules)
os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token: ')
!mkdir -p /content/dl /content/data
!kaggle competitions download -c daml-aitp-fall-26-challenge -p /content/dl
!cd /content/dl && unzip -q -o '*.zip' -d /content/data && ls /content/data

In [ ]:
# 3) locate the folder that holds train.csv (handles an extra nesting level) and verify layout
from pathlib import Path
hits = list(Path('/content/data').rglob('train.csv'))
assert hits, 'train.csv not found - check the download cell output'
root = hits[0].parent
os.environ['DATA_ROOT'] = str(root)
for need in ['classes.csv', 'sample_submission.csv', 'images/train', 'images/test']:
    print(need, 'OK' if (root / need).exists() else 'MISSING')
print('DATA_ROOT =', root)

In [ ]:
# 4) smoke test: baseline end to end (a few minutes on a GPU)
!python src/extract_features.py --model resnet50.a1_in1k
!python src/linear_probe.py --model resnet50.a1_in1k --name baseline
!python src/summarize.py